# Benchmark recordings

Evaluate selected regions against benchmark annotations and inspect POKER-DVS recordings. These experiments distinguish activity capture from target overlap.


## 1. Setup

Load local DVS benchmark HDF5 files and POKER-DVS recordings. Required paths are listed in docs/DATASETS.md.


In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import tonic.transforms as T

import foveanet as fn
import evaluate as ev
from tonic_convert import DVSBenchmark, PokerDVSCards

warnings.filterwarnings("ignore")

VOT_H5 = "data/DVSBENCH/INI_VOT_30fps_20160610.hdf5"
TRACK_H5 = "data/DVSBENCH/INI_TrackingDataset_30fps_20160610.hdf5"

vot = DVSBenchmark(VOT_H5)
track = DVSBenchmark(TRACK_H5)
cards = PokerDVSCards()

print("VOT Challenge 2015 :", vot)
print("TrackingDataset    :", track)
print("POKER-DVS          :", cards)
print()
print("VOT sequences   :", vot.data[:6], "...")
print("Track sequences :", track.data[:4], "...")

## 2. Ground-truth metrics

Compare selected-component IoU, best-component IoU and event share inside the annotated target. The best-component score separates clustering limitations from selection errors.


In [ ]:
seq = "bag"
events, boxes = DVSBenchmark(VOT_H5, sequences=[seq])[0]
x, y, t = events["x"], events["y"], events["t"]

print(f"sequence '{seq}': {len(events)} events over {(t.max() - t.min()) / 1e6:.1f} s")
print(f"ground-truth rows: {boxes.shape[0]}")
print(
    f"sensor coverage of the annotated box: "
    f"{100 * np.median([(ev.gt_box(r)[2] - ev.gt_box(r)[0]) * (ev.gt_box(r)[3] - ev.gt_box(r)[1]) for r in boxes]) / (240 * 180):.1f}%"
)

chosen, oracle, inside = ev.score_sequence(lambda: fn.PersistentGMM(), x, y, t, boxes, fn.WINDOW_US)
print()
print(f"frames scored          : {len(chosen)}")
print(f"events inside the truth: {100 * inside.mean():.0f}%")
print(f"chosen IoU             : {chosen.mean():.3f}")
print(f"oracle IoU             : {oracle.mean():.3f}")

## 3. Benchmark sequences

Score the selected sequences using the same frame window and selection rule. This sample is exploratory; notebook 06 contains the broader benchmark summary.


In [ ]:
def score_many(ds, names, window=fn.WINDOW_US):
    rows = []
    for nm in names:
        one = DVSBenchmark(ds.h5_path, sequences=[nm])
        e, b = one[0]
        c, o, i = ev.score_sequence(lambda: fn.PersistentGMM(), e["x"], e["y"], e["t"], b, window)
        rows.append((nm, i.mean(), c.mean(), o.mean(), (c > 0.5).mean()))
    return rows


rng = np.random.default_rng(0)
vot_pick = [vot.data[i] for i in rng.choice(len(vot.data), 10, replace=False)]
track_pick = [track.data[i] for i in rng.choice(len(track.data), 10, replace=False)]

for title, ds, names in [
    ("VOT Challenge 2015", vot, vot_pick),
    ("TrackingDataset", track, track_pick),
]:
    rows = score_many(ds, names)
    print(f"\n{title}")
    print(f"{'sequence':>28}{'in GT':>8}{'chosen':>9}{'oracle':>9}{'succ>0.5':>10}")
    for nm, i, c, o, s in rows:
        print(f"{nm[-28:]:>28}{100 * i:>7.0f}%{c:>9.3f}{o:>9.3f}{100 * s:>9.0f}%")
    a = np.array([r[1:] for r in rows])
    print(
        f"{'MEAN':>28}{100 * a[:, 0].mean():>7.0f}%{a[:, 1].mean():>9.3f}"
        f"{a[:, 2].mean():>9.3f}{100 * a[:, 3].mean():>9.0f}%"
    )

## Interpreting low overlap

High event activity does not imply target identity. Inspect the annotated target and background before attributing failures to a specific source.


## 4. Display modulation

Inspect the event-rate spectrum for periodic activity. A spectral peak alone does not show that display refresh dominates the tracking error; later measurements are recorded in FOVEANET_LOG.txt.


In [ ]:
x_v, y_v, t_v = events["x"], events["y"], events["t"]
print("dominant frequencies in the event rate, VOT 'bag':")
for hz, power in ev.dominant_frequencies(t_v):
    print(f"   {hz:7.1f} Hz   relative power {power:.2f}")

print("\ndoes denoising help?")
print(f"{'':>14}{'events':>10}{'in GT':>8}{'oracle IoU':>12}")
for label, tf in [("raw", None), ("Denoise", T.Denoise(filter_time=5_000))]:
    e2, b2 = DVSBenchmark(VOT_H5, sequences=[seq], transform=tf)[0]
    c2, o2, i2 = ev.score_sequence(
        lambda: fn.PersistentGMM(), e2["x"], e2["y"], e2["t"], b2, fn.WINDOW_US
    )
    print(f"{label:>14}{len(e2):>10}{100 * i2.mean():>7.0f}%{o2.mean():>12.3f}")

## 5. POKER-DVS

Inspect recording durations and event distributions in the original card recordings, rather than pre-cropped classification samples.


In [ ]:
for i, name in enumerate(cards.names):
    xc, yc, tc, pc = cards.load(cards.data[i])
    dur = (tc.max() - tc.min()) / 1e6
    print(
        f"{name}: {len(xc):>7} events, {dur:4.2f} s, {len(xc) / dur / 1000:>3.0f} keps, "
        f"x {xc.min()}-{xc.max()}, y {yc.min()}-{yc.max()}"
    )

print("\ndominant frequencies in cards_1 (no display, so no refresh peak expected):")
xc, yc, tc, pc = cards.load(cards.data[0])
for hz, power in ev.dominant_frequencies(tc, n=4):
    print(f"   {hz:7.1f} Hz   relative power {power:.2f}")

## 6. Frame windows on fast motion

Sweep frame duration and compare event capture with crop area. Longer windows can spread moving edges across a larger region.


In [ ]:
def card_sweep(x, y, t, window, k=fn.K, rule="persdens+hyst"):
    pg = fn.PersistentGMM(k=k)
    ids, caps, bfs = [], [], []
    for s, xs, ys, ts in fn.iter_frames(x, y, t, window=window, min_events=20):
        coords = np.column_stack([xs, ys]).astype(np.float32)
        pg.update(coords)
        cid = pg.select(rule=rule)
        b = pg.component_box(cid, coords)
        if b is None:
            continue
        ids.append(cid)
        caps.append(ev.events_inside(b, xs, ys))
        bfs.append(max((b[2] - b[0]) * (b[3] - b[1]), 1) / (128 * 128))
    if len(ids) < 5:
        return None
    d, l, sw = fn.dwell_stats(ids)
    return len(ids), d * window / 1e6, sw, np.mean(caps), np.mean(bfs)


print(f"{'window':>9}{'frames':>8}{'dwell s':>9}{'switch':>8}{'capture':>9}{'boxfrac':>9}")
for w in [2_000, 5_000, 10_000, 20_000, 33_000, 50_000]:
    r = card_sweep(xc, yc, tc, w)
    if r is None:
        print(f"{w / 1000:>8.0f}ms   too few frames")
        continue
    n, dwell_s, sw, cap, bf = r
    print(f"{w / 1000:>8.0f}ms{n:>8}{dwell_s:>9.2f}{100 * sw:>7.0f}%{cap:>9.2f}{bf:>9.3f}")

## 7. Selected card regions

Inspect crops at several times in the recording. These recordings do not provide the same target annotations as the tracking benchmark.


In [ ]:
SHOW_WINDOW = 10_000
pg = fn.PersistentGMM()
snaps = []
for s, xs, ys, ts in fn.iter_frames(xc, yc, tc, window=SHOW_WINDOW, min_events=20):
    coords = np.column_stack([xs, ys]).astype(np.float32)
    pg.update(coords)
    cid = pg.select(rule="persdens+hyst")
    b = pg.component_box(cid, coords)
    if b is None:
        continue
    snaps.append(
        ((s - tc.min()) / 1000, fn.frame_image(xs, ys), b, cid, pg.saliency_map(rule="persistence"))
    )

print(f"frames: {len(snaps)}")
pick = np.linspace(5, len(snaps) - 1, 7).astype(int)
fig, axes = plt.subplots(2, 7, figsize=(18, 5.2))
for col, i in enumerate(pick):
    ms, img, b, cid, smap = snaps[i]
    for row, (data, cmap) in enumerate([(img, "hot"), (smap, "magma")]):
        ax = axes[row, col]
        ax.imshow(data, cmap=cmap)
        ax.add_patch(
            patches.Rectangle(
                (b[0], b[1]), b[2] - b[0], b[3] - b[1], lw=1.8, edgecolor="cyan", facecolor="none"
            )
        )
        ax.axis("off")
        if row == 0:
            ax.set_title(f"{ms:.0f} ms", fontsize=9)
plt.suptitle("POKER-DVS cards_1 at 10 ms: events (top), persistence saliency (bottom)")
plt.tight_layout()
plt.show()

print(f"\n{'k':>4}{'frames':>8}{'dwell s':>9}{'switch':>8}{'capture':>9}{'boxfrac':>9}")
for k in [2, 3, 5, 8, 12]:
    r = card_sweep(xc, yc, tc, SHOW_WINDOW, k=k)
    if r:
        n, dwell_s, sw, cap, bf = r
        print(f"{k:>4}{n:>8}{dwell_s:>9.2f}{100 * sw:>7.0f}%{cap:>9.2f}{bf:>9.3f}")

## 8. Scope

Activity concentration, target overlap and classification accuracy are different measurements. Use the annotated benchmark and fixed-budget classifier comparisons for the corresponding claims.
